<a href="https://colab.research.google.com/github/sgisgeodata/sgis-data-manual/blob/main/%EA%B0%9C%EB%B3%84%20%EA%B5%90%EC%9C%A1%EC%9E%90%EB%A3%8C%20Training%20Materials/(260716)%20%EA%B4%91%EC%A3%BC%EC%97%B0%EA%B5%AC%EC%9B%90/GeoAI%20%EC%8B%A4%EC%8A%B5/GeoAI_water/geoai_gwangju.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GeoAI를 활용한 Sentinel-2 수체 탐지 실습

이번 실습에서는 Google Earth Engine에서 Sentinel-2 영상을 불러온 뒤, GeoAI 사전학습 모델을 이용해 수체 영역을 탐지합니다.

실습 전 다음 준비가 필요합니다.

- Google 계정 로그인
- Google Earth Engine 사용 등록
- 비상업·교육용 프로젝트 등록
- 새 프로젝트 만들기



## 1. 실습 환경 설치

Google Earth Engine과 지도 시각화에 필요한 패키지를 설치합니다.


In [ ]:
# 실습 패키지 설치
!pip install -q geemap earthengine-api geoai-py overturemaps

## 2. Google Earth Engine 인증

Google Earth Engine을 사용하기 위해 인증과 초기화를 수행합니다.  
`project` 값은 본인의 Earth Engine 프로젝트 ID에 맞게 수정합니다.
Google 계정 인증 창이 열리면 허용 버튼을 누릅니다.

In [ ]:
# 라이브러리 불러오기
import ee
import geemap

# Earth Engine 인증 및 초기화
ee.Authenticate(auth_mode="colab")
ee.Initialize(project="jm0629")

## 3. 실습 지역 ROI 설정

수체 탐지 실습에 사용할 관심 영역(ROI)을 설정합니다.  
여기서는 영월 청령포 일대의 사각형 영역을 사용합니다.


In [ ]:
# 실습 지역 ROI 설정
roi = ee.Geometry.Rectangle([128.435, 37.166, 128.451, 37.180])

## 4. Sentinel-2 영상 생성

Sentinel-2 L2A Surface Reflectance 영상을 불러오고, 구름이 적은 영상을 선택합니다.  
GeoAI 모델 입력에 맞도록 RGB+NIR 4개 밴드를 사용합니다.


In [ ]:
# Sentinel-2 구름 마스킹 함수
def mask_s2_clouds(image):
    qa = image.select("QA60")

    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    )

    return image.updateMask(mask)


# Sentinel-2 영상 컬렉션 생성
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(roi)
    .filterDate("2024-04-01", "2024-10-31")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
    .map(mask_s2_clouds)
)

# 구름 비율이 가장 낮은 영상 선택
best_image = ee.Image(s2.sort("CLOUDY_PIXEL_PERCENTAGE").first())

# 선택 영상 정보 확인
print(
    "선택된 영상 날짜:",
    ee.Date(best_image.get("system:time_start")).format("YYYY-MM-dd").getInfo()
)

print(
    "구름 비율:",
    best_image.get("CLOUDY_PIXEL_PERCENTAGE").getInfo()
)


# RGB+NIR 4밴드 구성
s2_rgbnir = (
    best_image
    .select(["B4", "B3", "B2", "B8"])  # Red, Green, Blue, NIR
    .clip(roi)
)

# GeoAI 모델 입력용 uint8 변환
s2_rgbnir_uint8 = (
    s2_rgbnir
    .clamp(0, 3000)
    .divide(3000)
    .multiply(255)
    .toUint8()
)

## 5. Sentinel-2 영상 시각화

생성한 Sentinel-2 영상을 지도 위에서 확인합니다.


In [ ]:
# Sentinel-2 RGB 영상 시각화
m = geemap.Map(center=[37.173, 128.443], zoom=15)

m.addLayer(
    s2_rgbnir_uint8,
    {
        "bands": ["B4", "B3", "B2"],
        "min": 0,
        "max": 255
    },
    "Sentinel-2 RGB"
)

m

## 6. Sentinel-2 영상 GeoTIFF 저장

GeoAI 모델에 입력하기 위해 Sentinel-2 영상을 GeoTIFF 파일로 저장합니다.


In [ ]:
# Sentinel-2 GeoTIFF 저장 경로
raster_path = "/content/cheongnyeongpo_sentinel2_rgbnir.tif"

# Sentinel-2 영상 내보내기
geemap.ee_export_image(
    s2_rgbnir_uint8,
    filename=raster_path,
    scale=10,
    region=roi,
    crs="EPSG:3857",
    file_per_band=False
)

## 8. GeoAI 수체 탐지 실행

GeoAI의 사전학습 수체 탐지 모델을 적용하여 수체 후보 영역을 추출합니다.


In [ ]:
# GeoAI 라이브러리 불러오기
import geoai

# 수체 탐지 결과 저장 경로
prediction_path = "water_prediction_005.tif"

# GeoAI 수체 탐지 모델 실행
geoai.object_detection(
    raster_path,                         # 입력 Sentinel-2 GeoTIFF
    prediction_path,                     # 탐지 결과 GeoTIFF
    model_path="water_detection.pth",    # 사전학습 수체 탐지 모델
    window_size=128,                     # 분할 처리 창 크기
    overlap=32,                          # 창 간 중첩 크기
    confidence_threshold=0.9,            # 탐지 신뢰도 기준
    batch_size=1,                        # 배치 크기
    num_channels=4,                      # 입력 영상 밴드 수
)

## 9. 수체 탐지 결과 벡터 변환

탐지 결과 TIFF를 GeoJSON 폴리곤으로 변환합니다.


In [ ]:
# 수체 탐지 결과 GeoJSON 변환
geojson_path = "water_prediction.geojson"

water_gdf = geoai.raster_to_vector(
    prediction_path,
    geojson_path,
    min_area=1,
    simplify_tolerance=1,
)

In [ ]:
print("수체 폴리곤 수:", len(water_gdf))

In [ ]:
water_gdf.head()

## 10. 수체 탐지 결과 시각화

원본 Sentinel-2 영상 위에 수체 탐지 결과를 중첩하여 확인합니다.


In [ ]:
raster_url = "https://raw.githubusercontent.com/sgisgeodata/sgis-data-manual/main/개별 교육자료 Training Materials/(260716) 광주연구원/GeoAI 실습/GeoAI_water/cheongnyeongpo_sentinel2_rgbnir.tif"
geoai.view_vector_interactive(
    water_gdf,
    tiles=raster_url
)

## 11. QGIS용 결과 저장

탐지 결과를 QGIS에서 열 수 있도록 GeoPackage 형식으로 저장합니다.


In [ ]:
# QGIS용 GeoPackage 저장
gpkg_path = "water_prediction.gpkg"

water_gdf.to_file(
    gpkg_path,
    layer="water_prediction",
    driver="GPKG"
)

In [ ]:
# Colab 파일 다운로드
from google.colab import files
files.download(gpkg_path)